# ETL Step 2 — Transform

Defines `transform(df)` — selects relevant columns, cleans nulls, standardizes types, and merges red/blue state classification. Returns `df_clean`.

In [1]:
import pandas as pd

def transform(df):
    # Select columns relevant to gender & political analysis
    keep_cols = [
    # Identity / posting metadata
    'ID',
    'LAST_UPDATED_DATE',
    'POSTED',

    # Job content
    'TITLE_CLEAN',          # replaces TITLE_NAME (which is dropped)
    'TITLE_RAW',
    'BODY',

    # Company
    'COMPANY',
    'COMPANY_NAME',
    'COMPANY_IS_STAFFING',

    # Education
    'EDUCATION_LEVELS',
    'EDUCATION_LEVELS_NAME',
    'MIN_EDULEVELS_NAME',
    'MAX_EDULEVELS_NAME',

    # Employment conditions
    'EMPLOYMENT_TYPE_NAME',
    'REMOTE_TYPE_NAME',
    'IS_INTERNSHIP',

    # Experience
    'MIN_YEARS_EXPERIENCE',
    'MAX_YEARS_EXPERIENCE',

    # Salary
    'SALARY_FROM',
    'SALARY_TO',

    # Geography (state dropped after red/blue join; city/MSA kept)
    'CITY_NAME',
    'MSA_NAME_INCOMING',

    # Industry — NAICS even levels only (2/4/6)
    'NAICS2',
    'NAICS2_NAME',
    'NAICS4',
    'NAICS4_NAME',
    'NAICS6',
    'NAICS6_NAME',

    # Skills
    'SKILLS',
    'SKILLS_NAME',
    'COMMON_SKILLS',
    'COMMON_SKILLS_NAME',
    'SOFTWARE_SKILLS',
    'SOFTWARE_SKILLS_NAME',

    # Occupation — legacy SOC broad + detailed (replaces SOC_2021_2 which is dropped)
    'SOC_2',
    'SOC_2_NAME',           # replaces SOC_2021_2_NAME
    'SOC_5',
    'SOC_5_NAME',
    'ONET',
    'ONET_NAME',

    # LOT taxonomy — career area level only
    'LOT_CAREER_AREA_NAME',
    'LOT_V6_CAREER_AREA',
    'LOT_V6_CAREER_AREA_NAME',
    'LOT_V6_OCCUPATION_GROUP',
    'LOT_V6_OCCUPATION_GROUP_NAME',

    # Lightcast sector
    'LIGHTCAST_SECTORS_NAME',

    # NAICS 2022 — even levels only (2/4/6)
    'NAICS_2022_2',
    'NAICS_2022_2_NAME',
    'NAICS_2022_4',
    'NAICS_2022_4_NAME',
    'NAICS_2022_6',
    'NAICS_2022_6_NAME',
]
    df_clean = df[keep_cols].copy()



    print("Shape after transform:", df_clean.shape)
    print("\nMissing values:\n", df_clean.isnull().sum())

    return df_clean

In [ ]:
import pandas as pd
df_raw = pd.read_csv('../data/raw/lightcast_2024.csv')
df_clean = transform(df_raw)

In [ ]:
df_clean.head(10)

In [ ]:
import sys
sys.path.insert(0, "..")  # project root so etl.load resolves
from etl.load import load

# Rename: two datasets in project (Lightcast + IPUMS) -- explicit prefix avoids confusion
df_lightcast_clean = df_clean.copy()

# Persist to disk -- downstream relational table splits read from this file
load(df_lightcast_clean, "../data/processed/lightcast_clean.csv")

In [ ]:
from collections import Counter
import ast
import pandas as pd

# Read raw file — only SKILLS_NAME column needed
df_raw = pd.read_csv("../data/raw/lightcast_job_postings.csv", usecols=["SKILLS_NAME"])

# Parse and count skills across all postings
all_skills = []
for entry in df_raw["SKILLS_NAME"].dropna():
    try:
        skills = ast.literal_eval(entry)
        all_skills.extend(skills)
    except:
        all_skills.extend([s.strip() for s in entry.split(",")])

top_skills = Counter(all_skills).most_common(20)
df_top_skills = pd.DataFrame(top_skills, columns=["Skill", "Count"])
df_top_skills.to_csv("../data/processed/lightcast_top_skills.csv", index=False)
print(f"Saved {len(df_top_skills)} skills to data/processed/lightcast_top_skills.csv")
df_top_skills